In [ ]:
# usual imports
import os
import numpy as np
from rail.utils.path_utils import find_rail_file
from rail.pipelines.estimation.somoclu_train_estimate import SOMTrainEstimatePipeline
from rail.core import common_params
import ceci

### Set common parameters for the photometric catalog

We use the small DC2-like test catalogs bundled with RAIL.
Their magnitude columns follow the `mag_{band}_lsst` / `mag_err_{band}_lsst`
naming convention, with a `photometry` HDF5 group and a `redshift` column.

In [ ]:
bands = 'ugrizy'
band_cols     = [f'mag_{b}_lsst'     for b in bands]
err_band_cols = [f'mag_err_{b}_lsst' for b in bands]

maglim_dict = {
    'mag_u_lsst': 24.0,
    'mag_g_lsst': 27.66,
    'mag_r_lsst': 27.25,
    'mag_i_lsst': 26.6,
    'mag_z_lsst': 26.24,
    'mag_y_lsst': 25.35,
}

common_params.set_param_defaults(
    bands=band_cols,
    err_bands=err_band_cols,
    nondetect_val=np.nan,
    ref_band='mag_i_lsst',
    redshift_col='redshift',
    mag_limits=maglim_dict,
    zmax=3.0,
)

### Configure the SOM informer and summarizer

In [ ]:
inform_dict = dict(
    hdf5_groupname='photometry',
    n_rows=10,
    n_columns=10,
    gridtype='hexagonal',
    std_coeff=1.0,
    som_learning_rate=0.3,
    n_epochs=2,
    ref_column_name='mag_i_lsst',
    initialization='random',
    column_usage='magandcolors',
)

summ_dict = dict(
    hdf5_groupname='photometry',
    spec_groupname='photometry',
    nzbins=301,
    nsamples=20,
    objid_name='id',
)

### Build the pipeline

In [ ]:
pipe = SOMTrainEstimatePipeline(inform_dict=inform_dict, summ_dict=summ_dict)

### Locate the input data files

Three separate catalogs are used:

| Role | File | Stage |
|------|------|-------|
| `input_train` | `training_100gal.hdf5` | `inform_som` — trains the SOM |
| `input_photo` | `validation_10gal.hdf5` | `summarize_som` — photometric catalog whose n(z) we want |
| `spec_input`  | `training_100gal.hdf5` | `summarize_som` — spectroscopic reference with true redshifts |

In [ ]:
train_file = find_rail_file('examples_data/testdata/training_100gal.hdf5')
photo_file = find_rail_file('examples_data/testdata/validation_10gal.hdf5')
spec_file  = find_rail_file('examples_data/testdata/training_100gal.hdf5')

output_dir = os.path.join('projects', 'som_test')

### Provide input files and initialise the pipeline

In [ ]:
input_dict = pipe.default_input_dict.copy()
input_dict.update(
    input_train=train_file,
    input_photo=photo_file,
    spec_input=spec_file,
)

In [ ]:
pipe_info = pipe.initialize(input_dict, dict(output_dir=output_dir, log_dir='.', resume=True), None)

In [ ]:
pipe.print_stages()

### Save the pipeline to a YAML file

In [ ]:
pipe.save('som_train_estimate.yml')

[For NERSC / cluster users!]

This won't work on a login node / Jupyter server without compute access. To run
the pipeline in batch, you need to:
1. Add `name: local` to the `site` section in `som_train_estimate.yml`.
2. SSH into a compute node, activate the RAIL environment, and run
   `ceci som_train_estimate.yml`.

In [ ]:
pr = ceci.Pipeline.read('som_train_estimate.yml')

In [ ]:
pr.run()